# create a simpler GPT

Following https://www.youtube.com/watch?v=kCc8FmEb1nY&list=WL&index=3&t=168s,
this notebook will create a Shakespear generator.

In [1]:
# get the data to train on
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-11 15:31:40--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.1s    

2026-09-11 15:31:41 (8.41 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r') as f:
    text = f.read()

In [3]:
print(f"length of dataset in characters: {len(text):,}")

length of dataset in characters: 1,115,394


In [4]:
print(text[:200])  # first 1000 characters

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [5]:
# get all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('"',''.join(chars), '"')
print(f'{vocab_size=}')


" 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz "
vocab_size=65


## tokenization
Character based

In [6]:
# create a mpping from to characters to integers
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda i: ''.join([itos[j] for j in i])

In [7]:
print(encode("hii there"))
print(decode(encode("hii there")))
print(encode(text[:10]))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there
[18, 47, 56, 57, 58, 1, 15, 47, 58, 47]


In [8]:
# using torch

import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(f'{data.shape[0]:,} {data.dtype}')
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

1,115,394 torch.int64


In [9]:
# create the X
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context}, the target is {target}")

When input is tensor([18]), the target is 47
When input is tensor([18, 47]), the target is 56
When input is tensor([18, 47, 56]), the target is 57
When input is tensor([18, 47, 56, 57]), the target is 58
When input is tensor([18, 47, 56, 57, 58]), the target is 1
When input is tensor([18, 47, 56, 57, 58,  1]), the target is 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]), the target is 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target is 58


In [57]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split =='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(xb)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"When input is {context}, the target is {target}")

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
When input is tensor([24]), the target is 43
When input is tensor([24, 43]), the target is 58
When input is tensor([24, 43, 58]), the target is 5
When input is tensor([24, 43, 58,  5]), the target is 57
When input is tensor([24, 43, 58,  5, 57]), the target is 1
When input is tensor([24, 43, 58,  5, 57,  1]), the target is 46
When input is tensor([24, 43, 58,  5, 57,  1, 46]), the target is 43
When input is tensor([24, 43, 58,  5, 57,  1, 46, 43]), the target is 39
When input is tensor([44]), the target is 53
When input is tensor([44, 53]), the target is 56
When input is tensor([44, 53, 56]), the target is 1
When input is tensor([44, 53, 56,  1]), the ta

## Create a bigram model

In [58]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) #(B, T, C)

        B, T, C = logits.shape
        logits2 = logits.view(B * T, C)
        if targets is not None:
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits2, targets)
        else:
            loss = None

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self.forward(idx)
            logits = logits[:, -1, :] # become (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx



In [62]:
m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 8, 65])
tensor(4.7459, grad_fn=<NllLossBackward0>)

&yuLFBFRtdxAqmu
Ypzx: r q.nqbwEJvA
&fWOEXdnrHvnb?T uNw&&LyuBhbIMRBYX-Hai:HRIOdtoe,fpNz.EGbwuV-S:js$f


In [63]:
# create optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % 1000 == 0:
        print(f"Step {steps}, Loss: {loss.item()}")

Step 0, Loss: 4.411225318908691
Step 100, Loss: 4.308027267456055
Step 200, Loss: 4.109256267547607
Step 300, Loss: 4.104085922241211
Step 400, Loss: 4.035861968994141
Step 500, Loss: 3.984624147415161
Step 600, Loss: 3.8701584339141846
Step 700, Loss: 3.7900290489196777
Step 800, Loss: 3.5847761631011963
Step 900, Loss: 3.6909291744232178
Step 1000, Loss: 3.6288962364196777
Step 1100, Loss: 3.5181679725646973
Step 1200, Loss: 3.4208335876464844
Step 1300, Loss: 3.37800931930542
Step 1400, Loss: 3.3044164180755615
Step 1500, Loss: 3.257460832595825
Step 1600, Loss: 3.1821448802948
Step 1700, Loss: 3.0923821926116943
Step 1800, Loss: 3.1692183017730713
Step 1900, Loss: 3.0254693031311035
Step 2000, Loss: 3.07540225982666
Step 2100, Loss: 3.085230827331543
Step 2200, Loss: 2.880385160446167
Step 2300, Loss: 2.9211459159851074
Step 2400, Loss: 2.931692361831665
Step 2500, Loss: 2.707077741622925
Step 2600, Loss: 2.8036112785339355
Step 2700, Loss: 2.7989797592163086
Step 2800, Loss: 2.931

In [68]:
print(decode(m.generate(idx, max_new_tokens=1000)[0].tolist()))


JUSe pooud hant-ery'lg chet aystl mere fit, ber.
I w halo s g:
BeayitiA:

S ay:

T:
So hauswhan, mesis y y theefrs nend l yos
THAny ha, avor oit, ilak, f usermake'sheie havot my tio ntrer st anomatyo nd sth w, RI,
ERqu'l;

RLoust migatou me te, chave.

UENIUD his
Amatonold ot s$ther wh ig paryGI th hy t f, ene? beand subrd YO:
F awifs cknddour cr:
IN w?
NGqulouredmeses r ber, hiedicrond ape is histh ve whif, shiorer:

PELAs w ou m th fus, If!
BEO:
E as dor hioupedso

Boocrr be Co ototo s dixourtato, h.
TI ge, t y ans think g;
The;And g'?ksth cechyointingat tice has hs sesaventle wito t be th tlllldur the,
AcDY:
FROHWhe sp.

BAno Fof prees d AMons tn w f o waprshe d?
BERENG hot.
LELBELYouke g, gonded cealf ars.
FOUS: ey in nghthistril I I baitst hathir's ave, se d wourdo, our:

Tug LELow' Frs ad I heowa; gdourele as ho ateereprprk, dum lyofHorut pellacotomithe:

Fro, Whe, be e Ifare.
Th ane YBY:
Torive verind!
A th, wangr MAnouavid foom ot RI: USIXdeme.
Lou RDES:
n wel,fo su, houre as,